# Sarvam-30B → GGUF Conversion Attempt

**Goal:** Convert `sarvamai/sarvam-30b` to GGUF format for local inference via Ollama/llama.cpp/LM Studio.

**Context:** Sarvam uses sigmoid routing (not softmax) in its MoE architecture. This is non-standard and may not be supported by llama.cpp's conversion script yet.

**Architecture:**
- 30B total params, 2.4B active
- 19 layers (1 dense + 18 MoE)
- 128 routed experts + 1 shared, top-6 routing
- Sigmoid routing (the blocker)

Read more: [mtrajan.substack.com/p/sarvam-open-is-not-sovereign](https://mtrajan.substack.com/p/sarvam-open-is-not-sovereign)

## Step 1: Setup

In [ ]:
!pip install -q huggingface_hub hf_transfer sentencepiece protobuf transformers torch numpy gguf

In [ ]:
# Clone llama.cpp
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp 2>/dev/null || (cd /content/llama.cpp && git pull)

# Build (CPU only — we just need the conversion tools)
!cd /content/llama.cpp && cmake -B build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=OFF && cmake --build build --config Release -j$(nproc) --target llama-quantize llama-cli

In [ ]:
# Check disk space (Colab gives ~100GB)
!df -h /content | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU'

## Step 2: Download Sarvam-30B

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="sarvamai/sarvam-30b",
    local_dir="/content/sarvam-30b",
    ignore_patterns=["*.bin"],  # prefer safetensors
)

## Step 3: Inspect Architecture

In [ ]:
import json

with open("/content/sarvam-30b/config.json") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))
print(f"\nModel type: {config.get('model_type', 'unknown')}")
print(f"Architectures: {config.get('architectures', [])}")

# Find MoE-specific keys
moe_keys = [k for k in config.keys() if any(x in k.lower() for x in ['expert', 'moe', 'router', 'routing'])]
print(f"MoE config keys: {moe_keys}")
for k in moe_keys:
    print(f"  {k}: {config[k]}")

## Step 4: Check llama.cpp Support

In [ ]:
# Does convert_hf_to_gguf.py know about Sarvam?
!grep -n 'sarvam\|sigmoid\|SarvamMoe' /content/llama.cpp/convert_hf_to_gguf.py || echo '\n>>> No Sarvam support found in conversion script'
print("---")
!grep -rn 'sarvam\|sigmoid' /content/llama.cpp/gguf-py/ 2>/dev/null | head -10 || echo '>>> No Sarvam support in gguf-py'
print("---")
!grep -rn 'sarvam\|sigmoid' /content/llama.cpp/src/ /content/llama.cpp/include/ 2>/dev/null | head -10 || echo '>>> No Sarvam support in llama.cpp source'

## Step 5: Attempt Conversion

This is the critical step. If Sarvam's `model_type` isn't recognized, it will fail here.

In [ ]:
!python3 /content/llama.cpp/convert_hf_to_gguf.py \
    /content/sarvam-30b \
    --outfile /content/sarvam-30b-f16.gguf \
    --outtype f16 \
    2>&1 | tee /content/conversion_log.txt

In [ ]:
# Check if it worked
import os
gguf_path = "/content/sarvam-30b-f16.gguf"
if os.path.exists(gguf_path):
    size_gb = os.path.getsize(gguf_path) / (1024**3)
    print(f"SUCCESS! GGUF created: {size_gb:.2f} GB")
    print("Proceeding to quantization...")
else:
    print("CONVERSION FAILED")
    print("\nFull log:")
    !cat /content/conversion_log.txt

## Step 6: Quantize (if conversion succeeded)

In [ ]:
import os
gguf_path = "/content/sarvam-30b-f16.gguf"

if os.path.exists(gguf_path):
    for method in ["q8_0", "q4_k_m"]:
        out = f"/content/sarvam-30b-{method}.gguf"
        print(f"\nQuantizing to {method}...")
        !cd /content/llama.cpp && ./build/bin/llama-quantize {gguf_path} {out} {method}
        if os.path.exists(out):
            print(f"  → {os.path.getsize(out)/(1024**3):.2f} GB")
else:
    print("No GGUF to quantize. See Step 5 for errors.")

## Step 7: Upload to HuggingFace (optional)

In [ ]:
# Uncomment and fill in to upload
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj="/content/sarvam-30b-q4_k_m.gguf",
#     path_in_repo="sarvam-30b-q4_k_m.gguf",
#     repo_id="YOUR_USERNAME/sarvam-30b-GGUF",
#     repo_type="model",
# )

## Step 8: If Conversion Failed — Diagnosis

If we got here, document what broke and why.

In [ ]:
# What model_type does Sarvam report?
import json
with open("/content/sarvam-30b/config.json") as f:
    config = json.load(f)

model_type = config.get("model_type", "unknown")
print(f"Sarvam model_type: '{model_type}'")

# What model_types does convert_hf_to_gguf.py support?
print("\nSupported model types in convert_hf_to_gguf.py:")
!grep -oP '"model_type".*?"\K[^"]+' /content/llama.cpp/convert_hf_to_gguf.py 2>/dev/null || \
 grep 'model_type' /content/llama.cpp/convert_hf_to_gguf.py | head -30

print(f"\n{'='*60}")
print("DIAGNOSIS:")
print(f"Sarvam uses model_type='{model_type}'")
print("If this type is not in the list above, that's the blocker.")
print(f"{'='*60}")
print("\nReferences:")
print("- vLLM PR #33942 (merged): Sarvam MoE support")
print("- llama.cpp PR #20275: sarvam_moe architecture (pending)")
print("- Blog: mtrajan.substack.com/p/sarvam-open-is-not-sovereign")